# Module 03 — Product structural change

*What each plant builds, and what those vehicles are.*

Modules 01 and 02 asked **where** automotive employment sits and **how** production is organised between plants. This module asks **what comes off the lines** — and answers it in two registers.

**The line-up.** Which models each plant builds, how often that line-up turns over, how many plants build the same model, and how far electrification has reached. This is a question about product *identity*.

**The vehicle.** What those models weigh, what they emit, and how both have moved. This is a question about product *substance*, and it is the part a model-count cannot see: a plant can build the same nameplates for a decade while the cars themselves gain 200 kg.

Two findings run through the whole module and are worth stating at the outset:

1. **Vehicles built in Europe got heavier — around +107 kg on a fixed set of plants between 2012 and 2023.** Roughly two-thirds of that is cars of every kind growing; only a third is the shift to heavier electric powertrains.
2. **The core builds the heavy cars.** German premium plants average 1,900–2,200 kg; Spanish, French and Italian plants 1,180–1,370 kg. The spatial division found in Module 01 is also a division of product.

---

In [ ]:
import os
%load_ext autoreload
%autoreload 2

import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import foundation as fd
import module02_organisational as m2
import module03_product as m3
import eea_crosswalk as xw
# --- archive bootstrap: resolve paths from the project root ---------------
# The original was CONFIG.data_dir = './data', which assumed the notebook ran
# beside a data/ folder. In the archived layout the inputs live in
# 02_data_inputs/, so the path is taken from specular_paths instead and this
# notebook works from wherever it is opened.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent))   # 01_code/
from specular_paths import P
fd.CONFIG.data_dir = str(P.inputs)
fd.CONFIG.out_dir  = str(P.outputs)
print('data:', fd.CONFIG.data_dir)
fd.CONFIG.out_dir  = './outputs'
print('data:', os.path.abspath(fd.CONFIG.data_dir))

---
# Part A — The line-up

*Which models each plant builds, and how that has changed.*

## A1. What vehicle attributes are actually available
This belongs in the **methods chapter** — it is the evidence for switching source. Panel (c) sizes the task: attributes for the top **150** models by frequency would cover **61%** of plant-model-years, top **200** → **71%**, top **300** → **86%**.

In [ ]:
ar = m3.attribute_readiness()
fig = m3.fig_attribute_readiness(ar, save=str(fd.CONFIG.out('fig03_attribute_readiness.png'))); plt.show()
display(ar['by_brand'][ar['by_brand']['models']>=5].round(0))

## A2. Electrification, identified from model names
ADEME's fuel field reaches 15% of the mix. But **dedicated BEV nameplates are identifiable by name** (ID.3, i4, EQS, e-tron, Mégane E-Tech…) at near-full coverage. The classifier is hand-verified against every candidate name in the panel.

The traps it must avoid are real and present in this data:

| name | reality |
|---|---|
| `C-Class EQ`, `GLC mit EQ-Technologie` | EQ **mild hybrid**, not BEV |
| `Punto EVO` | *evoluzione* — nothing electric |
| `Macan S`, `Macan GTS`, … | ICE; only `Macan 4 Electric` is BEV |
| `Electric Vehicle Drive Modules` | a **component**, not a car |

In [ ]:
for t in ['C-Class EQ','GLC mit EQ-Technologie','Punto EVO','Macan S','Auris Hybrid','Electric Vehicle Drive Modules']:
    assert not m3.classify_bev(t), t
for t in ['ID.3','i4 eDrive40','EQS','Q4 e-tron','e-Golf','Macan 4 Electric','Mégane E-Tech','Born']:
    assert m3.classify_bev(t), t
print('all spot-checks pass')

_ = m3.validate_bev_classifier()

### A2.1 How reliable is naming as a guide to electrification?
Precision on the overlap is 71%, and **both disagreements are ADEME matching errors, not classifier errors**:

* *Porsche Macan 4 Electric* → ADEME says **Petrol** (it matched the ICE Macan)
* *Ford Explorer EV* → ADEME says **Petrol plug-in hybrid** (it matched the US PHEV, not the Cologne-built BEV)

That is further evidence the ADEME layer is unreliable — and a reason to trust the nameplate route for the EV axis.

### A2.2 A dating error in the sources, and how it is caught
The source data puts the **Mercedes EQA at Rastatt in 2006–2012**. The EQA launched in **2021** — the press gap-filler assigned the plant's *current* model list to earlier years. Same for the BMW iX1 in 2013 (launched 2022).

`BEV_FIRST_YEAR` flags any BEV record pre-dating its nameplate's launch and sets `is_bev=False`, so the early electrification series is not fictitious. Records are **flagged, not deleted** (`bev_implausible`).

In [ ]:
mp = m3.model_panel()
print('implausible BEV records neutralised:', int(mp['bev_implausible'].sum()))
display(mp[mp['bev_implausible']][['plant_name','year','model','value_source']].head())
print('\nfirst BEV year after the guard:', int(mp[mp['is_bev']]['year'].min()), '(BMW i3 — correct)')

In [ ]:
fig = m3.fig_electrification(mp, save=str(fd.CONFIG.out('fig03_electrification.png'))); plt.show()
g = mp.groupby('year')['is_bev'].agg(['sum','size']); g['bev_pct'] = (100*g['sum']/g['size']).round(1)
display(g.tail(12))

## A3. Model line-ups and how fast they turn over
Portfolio size, churn (models added/dropped), renewal rate — the product-cycle pulse of the network.

In [ ]:
mix = m3.plant_model_mix(mp)
fig = m3.fig_model_mix(mix, save=str(fd.CONFIG.out('fig03_model_mix.png'))); plt.show()
display(mix.groupby('year')[['n_models','n_bev','bev_share']].mean().round(2).tail(8))

In [ ]:
ch = m3.model_churn(mp)
fig = m3.fig_churn(ch, save=str(fd.CONFIG.out('fig03_churn.png'))); plt.show()
print('most-churned plants (total model entries):')
display(ch.groupby(['plant_name','country_iso2'])['entries'].sum().sort_values(ascending=False).head(10))
ch.to_csv('outputs/tbl03_model_churn.csv', index=False)

## A4. Models built in more than one plant
How many plants build a given model at once — a footprint-strategy indicator. Rising values mean flexibility, or a transfer in progress.

In [ ]:
ms = m3.multi_sourcing(mp)
fig = m3.fig_multi_sourcing(ms, save=str(fd.CONFIG.out('fig03_multi_sourcing.png'))); plt.show()
print('most multi-sourced models (max plants at once):')
display(ms.groupby('model')['n_plants'].max().sort_values(ascending=False).head(10))

## A5. Employment alongside the model line-up
On the strictly balanced universe (31 plants, 2012–2023). These are **associations**, not effects; nothing here identifies causality and the thesis does not claim it.

Note especially panel (c): it compares two **changing** sets of plants — a plant moves between them the year it starts building a BEV. Read it as composition, not as an effect of electrification.

In [ ]:
ev = m3.employment_vs_mix(window=(2012, 2023))
fig = m3.fig_employment_vs_mix(ev, save=str(fd.CONFIG.out('fig03_employment_vs_mix.png'))); plt.show()
print('correlation (plant-years, descriptive):')
print(ev[['employees','n_models','n_bev','bev_share']].corr().round(2).to_string())

---
## What Module 03 establishes

* **BEV share of the model mix rises from 0% (pre-2013) to 9.3% (2025)**, starting with the BMW i3 in 2013 — measured at near-full coverage from nameplates, where the attribute route would have reached 15%.
* **Portfolio size grew** from ~4.5 to ~8.3 models per plant: plants build more distinct models than they did.
* Model **entries consistently exceed exits** — the network is adding nameplates, not just rotating them.
* The **attribute-readiness diagnostic** is a methods-chapter figure in its own right, and it quantifies the target for the replacement source.

## Open items

1. **Module 03b — mass.** Download the EEA CO2 monitoring database, match on make + commercial name, and redo the size analysis. Target ≥200 models (≈71% of plant-model-years).
2. **Churn is partly a coverage artefact.** A model 'enters' when it first appears in that plant's sources. Worth restricting churn analysis to well-covered plant-years before drawing conclusions.
3. **`BEV_FIRST_YEAR` should be extended** as more BEV nameplates enter the panel; it currently covers those present.
4. The **back-projection bug** in `established_press` may affect non-BEV models too — worth a systematic check of model start-years against launch dates.

**Next — Module 04: macro–micro integration**, using the plant→NUTS crosswalk to place plants inside their regional context and measure what share of each region's NACE 29 employment they represent.

---
# Part B — The vehicle

*What those models weigh and what they emit.*

Part A identified models by name. This part attaches a physical measurement to each of them, taken from the European registration data collected under Regulation (EU) 2019/631, which records every new car registered in the eleven countries that host the plants studied here.

## B1. Matching model names to the registration data
EEA's commercial-name field is free text typed by manufacturers: one car appears as `A 180`, `A180`, `A 180 CDI`, `A 180 D 4MATIC`. Our side carries generation codes EEA never uses (`GOLF MK2`, `A3 8V`). An exact join reaches 48%.

`eea_crosswalk` normalises both sides to a model stem and matches on it, aggregating variants with registration weights — which is also the statistic we want: *the mass of the Golf* is the weighted mean over all Golf versions sold.

In [ ]:
mp = m3.model_panel()
cw = xw.build_crosswalk(mp)
rep = xw.match_report(mp, cw)

display(rep)
print(f"coverage {rep.attrs['coverage_pct']:.1f}%  "
      f"(precise {rep.attrs['precise_pct']:.1f}%, remainder coarse but valid)")

# sanity: do well-known models come out at plausible masses?
for b, m in [('Volkswagen','Golf'), ('Volkswagen','Polo'), ('Porsche','911'),
             ('Mercedes-Benz','S-Class'), ('Renault','Clio'), ('BMW','X5')]:
    r = cw[(cw.brand == b) & (cw.model == m)]
    if len(r):
        print(f'  {b:14s} {m:10s} {r.mass_kg.iloc[0]:7.0f} kg   ({r.rule.iloc[0]})')

## B2. How heavy are the vehicles each plant builds?
**The weighting problem, stated plainly.** We know which models a plant builds and what each weighs. We do *not* know how many of each it built — `production_value` is a plant total, and `production_volume_per_model` exists in the source schema as an empty column that was never populated.

So a plant average has to assume something about the mix inside it. Two assumptions bracket the answer and both are computed:

| | assumption |
|---|---|
| `equal` | every model the plant builds counts the same |
| `registrations` | models weighted by their European registrations |

Neither is right. The gap between them is the honest size of the uncertainty.

In [ ]:
pmm = m3.plant_model_mass()
pm  = m3.plant_mass(pmm, weighting='equal')

print(f'{pmm.plant_uid.nunique()} assembly plants, '
      f"{100*pmm.mass_kg.notna().mean():.0f}% of model rows carry a mass")
display(m3.coverage_report(pmm).tail(6).round(1))

fig = m3.fig_plant_mass(pm, year=2023, save=str(fd.CONFIG.out('fig03b_plant_mass.png'))); plt.show()

> The ordering is the Module 01 story told through the product. German premium plants — Munich, Emden, Leipzig, Sindelfingen, Neckarsulm, Dingolfing — sit at 1,900–2,200 kg; Spanish, French and Italian plants sit at 1,180–1,370 kg. The core keeps the large, high-value vehicles. That is **functional specialisation**, not simply relocation.

## B3. Fixing the set of plants before reading any trend
The raw production-weighted series rests on **5 plants in 2006, 22 in 2016 and 12 in 2025**. A trend over that mostly records which plants happened to publish an output figure. In Module 02 the same artefact manufactured two findings that did not survive balancing.

Choose the window on the evidence, not by convention.

In [ ]:
print('Requiring an observed production figure:')
display(m3.balance_report(pmm, 'production'))

print('Requiring only a model list:')
display(m3.balance_report(pmm, 'equal'))

> **Production weighting cannot be balanced.** Only 5–6 plants report an output figure in every year of any usable window, against 31 that report a model list over 2012–2023. So the headline series weights plants equally on a balanced panel, and the production-weighted series is shown only as a caution. This is a limit of the source, and it is stated rather than worked around.

## B4. Each plant's vehicles, followed through time
Section 2 is a single-year cross-section. This is the same measure followed **through time**, on the balanced panel — the view in which a plant switching product shows up.

> Ford Genk closed in December 2014 but its Wikipedia model list persists to 2025, and because those stale rows are *complete* they survive panel balancing. `plant_model_mass()` drops model-years recorded after a known closure by default; it prints a warning saying which.

In [ ]:
tr = m3.plant_trajectories(pmm, window=(2012, 2023))
fig = m3.fig_plant_trajectories(pmm, window=(2012, 2023),
                                 save=str(fd.CONFIG.out('fig03b_plant_trajectories.png'))); plt.show()

display(tr[['plant_name','country_iso2','mass_y0','mass_y1','change_kg','change_pct']].round(1))

> The large moves are **product switches, not drift**. Audi Brussels +924 kg (A1 hatchback → Q8 e-tron SUV), Zwickau +361 (Golf → ID.3/ID.4), Cologne +360 (Fiesta → electric SUV), Dresden −465 (Phaeton → ID.3). Twenty of thirty plants build heavier cars than in 2012, median +65 kg.

> Several plants sit at exactly 0: their recorded portfolio did not change over the window. That is a statement about the *source*, which records model presence from article snapshots, not proof that nothing changed on the line.

## B5. The European trend

In [ ]:
fig = m3.fig_mass_trend(window=(2012, 2023),
                         save=str(fd.CONFIG.out('fig03b_mass_trend.png'))); plt.show()

t = m3.mass_trend(pmm, 'equal', 'equal', window=(2012, 2023))
display(t[['year','mean_mass_kg','n_plants']].round(0))

## B6. Cars getting bigger, or powertrains changing?
Measured on EEA registrations directly rather than through our plant sample, because this is a statement about the European fleet.

The decomposition is **symmetric (Törnqvist)**: each class's mass change is weighted by its average share, each class's share change by its average mass. A naive version anchored at 2010 gives battery cars — which barely existed then — the same baseline as everything else, and the composition term collapses to zero by construction.

In [ ]:
dec = m3.powertrain_decomposition()
fig = m3.fig_powertrain(dec, save=str(fd.CONFIG.out('fig03b_powertrain.png'))); plt.show()

r = dec.iloc[-1]
print(f"{dec.attrs['y0']} -> {dec.attrs['y1']}:  total {r.total_change:+.0f} kg")
print(f"   cars of each type getting heavier : {r.within_effect:+6.0f} kg")
print(f"   shift to heavier powertrains      : {r.between_effect:+6.0f} kg")
print(f"   unexplained interaction           : {r.interaction:+6.0f} kg"
      f"  ({abs(100*r.interaction/r.total_change):.0f}% of total)")

## B7. How much does the unknown model mix matter?

In [ ]:
sens = m3.weighting_sensitivity(pmm)
fig = m3.fig_weighting_sensitivity(sens,
                                    save=str(fd.CONFIG.out('fig03b_sensitivity.png'))); plt.show()

print(f'median gap {sens.gap_pct.median():+.1f}%  |  '
      f'90th pct {sens.gap_pct.abs().quantile(.9):.1f}%  |  '
      f'max {sens.gap_pct.abs().max():.1f}%')

ex = m3.exact_mass_plants(pmm)
print(f'\n{len(ex)} plant-years built a single model, so their average mass '
      'needs no assumption at all.')
display(ex[['plant_name','year','model','mass_kg','production_value']].head(8))

> These single-model rows **cannot** validate the weighting choice — with one model both schemes give the same answer, so they carry no information about which is better. They are the rows where the *level* is certain, and that is all they are used for.

## B8. Following a model's own trajectory
Everything above uses one mass per model for the whole period. That is defensible for mass — a Golf is roughly a Golf's weight throughout — but **wrong for CO2**, which fell by a quarter over the window and does not exist before 2018.

`build_crosswalk(by_year=True)` collapses the matched EEA rows *per year* instead, so a model carries its own trajectory.

In [ ]:
pmy = m3.plant_model_mass(by_year=True)

print(f'yearly panel: {len(pmy):,} rows | with mass {pmy.mass_kg.notna().sum():,}'
      f' | with CO2 {pmy.co2_g_km.notna().sum():,}')

cy = xw.build_crosswalk(mp, by_year=True)
display(cy[(cy.brand=='Volkswagen') & (cy.model=='Golf')]
        [['year','mass_kg','co2_g_km','bev_share']].tail(8).round(1))

## B9. Tailpipe emissions per plant
**Two limits, both structural.**

*The measure.* This is tailpipe CO2 under Reg. (EU) 2019/631, **not life-cycle**. A battery car is 0 g/km by regulatory definition — the emissions of building its battery and generating its electricity sit outside the measure entirely. A plant that electrifies falls to zero here while its real footprint does not.

*The window.* WLTP replaced NEDC across 2017–2019, so the field is empty before 2018. The aggregator previously divided a NaN-summed numerator by *total* registrations and returned a confident **0.0 g/km for 2010–2017** — "not measured" rendered as "zero emissions", which are opposites. It now carries a separate CO2 denominator and returns NaN. The series starts in 2019.

In [ ]:
co2 = m3.plant_co2(pmy)
fig = m3.fig_plant_co2(co2, year=2023, save=str(fd.CONFIG.out('fig03b_co2.png'))); plt.show()

display(co2.groupby('year').agg(plants=('plant_uid','nunique'),
                                mean_co2=('mean_co2_g_km','mean')).round(1))

flagged = co2[co2.all_electric_recorded]
print(f'{len(flagged)} plant-years read 0 g/km because every model in their '
      'RECORDED portfolio is electric:')
display(flagged[['plant_name','year','n_models_co2','n_models_listed']].head(8))

> ⚠ **Check these before quoting any single plant.** BMW Munich reads 0 g/km for 2023 because its recorded model list that year is four i4 variants — it plainly also built the 3-series. The model list comes from article snapshots and can miss combustion models still in production. The flag says *verify*, it does not delete the row.

## B10. Did the plants that electrified also get heavier?
The fleet-level decomposition in section 5 said cars growing (+120 kg) mattered more than the powertrain shift (+61 kg). This asks the same question **across plants**.

In [ ]:
ev = m3.electrification_vs_mass(pmy, window=(2012, 2023))
fig = m3.fig_electrification_vs_mass(ev,
                                      save=str(fd.CONFIG.out('fig03b_evmass.png'))); plt.show()

print(f"{ev.attrs['n_plants']} plants | Pearson r = {ev.attrs['pearson_r']:.2f}"
      f" | Spearman rho = {ev.attrs['spearman_r']:.2f}")
display(ev[['plant_name','country_iso2','d_bev_pp','d_mass_kg']].round(1))

> **The association is weak — r = 0.10.** Plants that electrified more did gain more mass (median +233 kg against +102 kg for those that did not electrify at all), but the twelve plants with *no* change in BEV share still gained a median **+102 kg**. Mass rose where electrification did not happen, which is the plant-level echo of the fleet-level result.

> **Descriptive only.** Twenty plants, no controls, and both axes are computed from the same model list — a plant adding an electric SUV moves on both by construction. This is an association worth reporting and not an effect worth estimating.

## B11. Mass by bloc: the core, and the two peripheries

Module 01 showed employment shifting from the core toward the eastern periphery. This asks whether the two also differ in *what* they build — whether the core holds the large, high-value vehicles while the periphery takes the smaller ones. If it does, the reallocation is a division of labour and not only a change of address.

In [ ]:
bloc = m3.mass_by_bloc(pmm, window=(2012, 2023))
fig = m3.fig_mass_by_bloc(bloc, save=str(fd.CONFIG.out('fig03_mass_by_bloc.png'))); plt.show()

display(bloc.pivot(index='year', columns='bloc', values='mean_mass_kg').round(0))

> **Read this one carefully.** The core rises from 1,564 to 1,655 kg while the southern periphery falls from 1,460 to 1,366 — consistent with the core holding larger vehicles. But the peripheral blocs rest on **four plants each**, so a single plant entering or leaving moves the line. Treat it as suggestive and say so in the text; the plant-level chart in section 2 carries the same finding on firmer ground.

## Save the tables

In [ ]:
cw.to_csv('outputs/tbl03b_eea_crosswalk.csv', index=False)
cy.to_csv('outputs/tbl03b_eea_crosswalk_yearly.csv', index=False)
pm.to_csv('outputs/tbl03b_plant_mass.csv', index=False)
co2.to_csv('outputs/tbl03b_plant_co2.csv', index=False)
ev.to_csv('outputs/tbl03b_electrification_vs_mass.csv', index=False)
dec.to_csv('outputs/tbl03b_powertrain_decomposition.csv', index=False)
sens.to_csv('outputs/tbl03b_weighting_sensitivity.csv', index=False)
print('saved to', os.path.abspath('outputs'))